# LUMAE M1 Product Ads — Colab fine-tuning
Run on a CUDA Colab runtime after uploading the portable LUMAE bundle and obtaining the trusted official Moment-DETR CLIP-only checkpoint. The 30 AI intervals are unverified pseudo labels; all have LOW semantic confidence and zero ranking margin. Validation and test contain only pseudo labels. Saliency GT is absent. No test labels are used for checkpoint selection.

## Transfer the frozen input bundle
On the local machine, run `python -m experiments.M1_lumae_ads.train_frozen package --lumae-root . --bundle-path local_data/exports/lumae_ads_v1_colab.zip`. Upload that ZIP to Google Drive. Open this notebook in Colab, set `BUNDLE` to its Drive path, then run the next cell. The ZIP contains the runner, frozen manifests, source CSVs, and 48 videos.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import drive
drive.mount('/content/drive')
BUNDLE = Path('/content/drive/MyDrive/lumae_ads_v1_colab.zip')  # Adjust to your upload path.
assert BUNDLE.is_file(), f'Missing uploaded bundle: {BUNDLE}'
with ZipFile(BUNDLE) as archive:
    archive.extractall('/content/lumae')
print('Extracted frozen inputs to /content/lumae')

In [ ]:
from pathlib import Path
import sys

LUMAE_ROOT = Path('/content/lumae') if Path('/content/lumae').is_dir() else Path.cwd()
MOMENT_ROOT = Path('/content/moment_detr')
PARENT_CKPT = MOMENT_ROOT / 'run_on_video/moment_detr_ckpt/model_best.ckpt'
FEATURE_ROOT = Path('/content/m1_features')
RESULTS_ROOT = Path('/content/m1_results')
sys.path.insert(0, str(LUMAE_ROOT))
from experiments.M1_lumae_ads.train_frozen import preflight, extract_features, train as train_m1
handoff, manifest, splits = preflight(LUMAE_ROOT)
print({name: len(splits[name]) for name in ('train', 'val', 'test')})
print('Dataset SHA-256:', manifest['dataset_sha256'])
print('Validation provenance: AI_PSEUDO_LABELED_ONLY')

## Upstream environment
Clone the official Moment-DETR repository at `MOMENT_ROOT` and provide its CLIP-only pretrained checkpoint at `PARENT_CKPT`. The next cells install Python dependencies and check prerequisites before spending GPU time. Use a trusted checkpoint because loading its saved Python options requires deserialization.

In [ ]:
import subprocess
if not MOMENT_ROOT.is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/jayleicn/moment_detr.git', str(MOMENT_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'ffmpeg-python', 'ftfy', 'regex', 'easydict', 'tensorboard', 'tabulate', 'scikit-learn', 'pandas'], check=True)

In [ ]:
import shutil
import torch
assert torch.cuda.is_available(), 'Select a CUDA Colab runtime'
assert (MOMENT_ROOT / 'moment_detr/model.py').is_file(), 'Clone the official Moment-DETR checkout'
assert PARENT_CKPT.is_file(), 'Provide the trusted CLIP-only checkpoint'
assert shutil.which('ffmpeg') and shutil.which('ffprobe'), 'Install ffmpeg'
print('CUDA:', torch.cuda.get_device_name(0))

## Extract CLIP features
This uses the upstream Moment-DETR `ClipFeatureExtractor` at one frame per two seconds. It extracts features for train and validation only, checks shapes, and records hashes. Use a fresh `FEATURE_ROOT` for each extraction.

In [ ]:
features = extract_features(LUMAE_ROOT, MOMENT_ROOT, FEATURE_ROOT)
print('Video features:', len(features['video']), 'Text features:', len(features['text']))

## Fine-tune temporal spans
The runner loads the trusted pretrained weights, trains on 34 frozen train rows, and selects a checkpoint on seven pseudo-labeled validation rows. It omits saliency targets and sets their loss weight to zero. A new results directory receives `model_best.ckpt`, `history.json`, and `result.json`. These validation metrics are pseudo-label agreement only; no human-grounded performance result is claimed.

In [ ]:
result = train_m1(
    LUMAE_ROOT, MOMENT_ROOT, PARENT_CKPT, FEATURE_ROOT, RESULTS_ROOT,
    epochs=50, batch_size=4, learning_rate=1e-5, patience=10,
)
print(result)